# Agents

#### Where we are

You've now built every piece:

```
Phase 1   →  how LLMs predict text
Phase 2-4 →  embeddings, vector DB, RAG (give the model knowledge)
Phase 5   →  prompting (control behavior)
Phase 6   →  APIs + structured outputs (talk to it from code)
Phase 7   →  tool calling (let it act)
Phase 8   →  evaluation (know if it works)
```

In Phase 7 we ended with a one-liner:

***An agent is basically an LLM that can use tools.***

Now we make it precise. An agent is:

```
An LLM
  +  a set of tools
  +  memory
  +  a LOOP that lets it decide its own next step
```

The difference from a plain tool call is **autonomy**: instead of you scripting the steps, the model decides — step by step — what to do until the goal is reached.

```
Tool calling  →  you control the flow
Agent         →  the model controls the flow
```

### The Agent Loop (ReAct)

The core pattern has a name: **ReAct** = *Reason + Act*.

```
Reason        "What do I need to do next?"
  ↓
Choose Tool   "I'll call get_weather('London')"
  ↓
Observe       "Result: 14°C, raining"
  ↓
Reason Again  "Now I can answer / or I need another tool"
  ↓
... repeat until the goal is met ...
  ↓
Final Answer
```

#### Why a loop, not a single call?

Real tasks need several steps, and the model can't know step 3 until it sees the result of step 2.

```
"Email our top customer a weather warning for their city."

  Reason → get_top_customer()        → "Acme Corp, city: London"
  Reason → get_weather("London")     → "storm warning"
  Reason → send_email(acme, "...")    → "sent"
  Reason → done → "Done. Warned Acme Corp about the London storm."
```

Each loop, the model sees everything so far (the conversation + tool results) and decides the next action. That accumulated history is its **working memory**.

### Tool Selection

With multiple tools available, the agent must pick the right one each step. It decides using:

```
- the tool NAME + DESCRIPTION (Phase 7)
- the current goal
- what it has already observed
```

```
Goal: "What's 18% tax on customer 42's total orders?"

Step 1: needs order data   → get_customer_orders(42)
Step 2: needs math         → calculator("750 * 0.18")
Step 3: has everything     → answer
```

#### Good tool selection depends on good design

```
✓ Few, well-named tools with clear descriptions
✓ Non-overlapping purposes (no two tools that "kind of" do the same thing)
✗ 50 vague tools → the model picks wrong, loops, or wastes tokens
```

Tool selection quality is something you should **evaluate** (Phase 8): did the agent choose the right tools in the right order?

### Memory

An agent needs to remember things — but there are two very different kinds.

```
Short-term memory (working memory)
  = the current conversation + tool results in the context window
  - lives in the prompt
  - limited by context size + token cost (Phase 1)
  - lost when the context is cleared
```

```
Long-term memory
  = facts stored OUTSIDE the model, retrieved when needed
  - "the user prefers metric units"
  - "last week we resolved ticket #55 this way"
  - stored in a database / vector DB and pulled back via RAG (Phase 4)
```

#### How they work together

```
Long-term store (vector DB)
        ↓ retrieve relevant memories
Short-term context  ←  + current conversation + tool results
        ↓
       LLM reasons
```

So "agent memory" is mostly the techniques you already learned:

```
Short-term  →  context window management
Long-term   →  embeddings + vector DB + retrieval
```

This is why we taught RAG and vector databases *before* agents — agent memory is RAG pointed at the agent's own history.

### Planning

For complex goals, strong agents **plan before acting** instead of reacting one step at a time.

```
Reactive (basic ReAct):
  decide ONE step → act → decide next → ...

Planned:
  1. break the goal into a list of sub-steps
  2. execute them (calling tools as needed)
  3. adjust the plan if something fails
```

Example:

```
Goal: "Prepare a weekly sales report and email it to the manager."

Plan:
  1. query sales for the last 7 days
  2. compute totals and top products
  3. format a summary
  4. email it to the manager

Then execute step by step, re-planning if a step fails.
```

```
Planning = think first, then act.
Helps with multi-step tasks; reduces wandering and wasted tool calls.
```

### A Full Worked Example

```
User: "Is it a good idea to ship to our top customer today?"

┌─ Reason:  I need to know who the top customer is.
│  Act:     get_top_customer()
│  Observe: { name: "Acme", city: "London" }
│
├─ Reason:  Shipping depends on weather. Check London.
│  Act:     get_weather("London")
│  Observe: { condition: "severe storm" }
│
├─ Reason:  A storm is risky. Check the courier status too.
│  Act:     get_courier_status("London")
│  Observe: { status: "delays expected" }
│
└─ Reason:  Storm + courier delays → advise against shipping today.
   Final Answer:
   "I'd hold the shipment. London has a severe storm and the
    courier reports delays. Recommend shipping tomorrow."
```

Notice what happened:

```
- multiple tools, chosen dynamically (tool selection)
- each step used the previous result (memory)
- the order was decided by the model, not by you (autonomy)
- it reasoned to a recommendation, not just raw data
```

That is an agent.

### Risks & Guardrails

Autonomy is powerful and dangerous. Things that go wrong with agents:

```
✗ Infinite loops        →  it keeps calling tools, never finishes
✗ Runaway cost          →  every loop = more tokens = more money
✗ Wrong tool / bad args →  it calls delete_user() when it shouldn't
✗ Hallucinated calls    →  invents a tool or arguments
✗ Compounding errors    →  a wrong step 2 poisons steps 3-5
✗ Prompt injection      →  tool output contains "ignore instructions..."
```

Guardrails you should always add:

```
✓ Max steps / max loops limit
✓ Budget / token cap per run
✓ Human approval for destructive actions (delete, pay, email)
✓ Validate tool arguments before executing (Phase 6)
✓ Treat tool output as untrusted (Phase 5 injection)
✓ Log every step so you can evaluate it (Phase 8)
```

```
An agent without guardrails is a script with no limits
running on your production systems. Design accordingly.
```

### Summary & Goal

```
Agent          =  LLM + tools + memory + a decision loop
Agent Loop     =  Reason → Choose Tool → Observe → Reason Again  (ReAct)
Tool Selection =  picking the right tool each step
Memory         =  short-term (context) + long-term (vector DB / RAG)
Planning       =  break the goal into steps before acting
Guardrails     =  step limits, budgets, approvals, validation, logging
```

#### Goal

***Understand agents as an LLM that decides its own steps in a loop, using tools and memory.***

You can build a basic agent with a `while` loop and the tool-calling code from Phase 7. But as agents grow — branching, retries, multiple agents, persistence — that hand-rolled loop gets messy.

That's the problem **LangGraph** solves, and **MCP** standardizes how agents connect to tools — the final **Phase 10**.